# KIRC classification of control, early or late stage

In [ ]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '112'
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.express as px
from src.classification import classification_benchmark

In [ ]:
# Set working directory to repository root
wd='/mnt/c/Repos/evenflow_cancer'

In [ ]:
# Load preprocessed data
preprocessed_data = pd.read_csv(os.path.join(wd,'data/interim/preprocessed/rnaseq_maha.csv'),index_col=0)
preprocessed_data.shape

In [ ]:
# Load preprocessed metadata
preprocessed_metadata = pd.read_csv(os.path.join(wd,'data/interim/preprocessed/clinical_subset.csv'),index_col=0)
preprocessed_metadata.shape

In [ ]:
# Check data dimensions (patients in rows, genes in columns)
if not preprocessed_data.shape[0]==preprocessed_metadata.shape[0]:
    preprocessed_data=preprocessed_data.T
print(preprocessed_data.shape,preprocessed_metadata.shape)

In [ ]:
stages = preprocessed_metadata['ajcc_pathologic_tumor_stage']
stages.value_counts()

# Classification

## Early, late

In [ ]:
# Create directory to save results
date = '20250919'
save_dir = os.path.join(wd,f'reports/figures/{date}_classifying_kirc')
os.makedirs(save_dir,exist_ok=True)

In [ ]:
# Classification on weighted data with different seeds
seeds = np.random.randint(0,1e9,10)
metrics_list_real_all_genes = []
model_real_all_genes = []
for seed_i in tqdm(seeds):
    weighted_classification = classification_benchmark(
        X_data=preprocessed_data,
        y_data=stages,
        classification_type='weighted',
        num_classes=2,
        seed=seed_i,
        test_size=0.2,
        n_br=1, # for testing, otherwise use 100
        num_threads=4,
        n_trials=1, # for testing, otherwise use 100
    )
    (model, metrics_weights, y_test_le, y_pred_weights, data_weights,weighted_params) = weighted_classification
    metrics_list_real_all_genes.append(metrics_weights)
    model_real_all_genes.append(model)

In [ ]:
# Adapt dataframe with metrics for plotly
df_classification_results=pd.concat(metrics_list_real_all_genes)
df_classification_results.reset_index(inplace=True)
df_classification_results.drop(columns="index",inplace=True)
df_classification_results.drop(index=np.arange(0,20,2),inplace=True)
df_classification_results

In [ ]:
# --- Data Preparation ---
df_melted = df_classification_results.melt(var_name='Metric', value_name='Score')

# --- Create Plot ---
fig = px.box(
    df_melted,
    x='Metric',
    y='Score',
    color='Metric',
    labels={'Score': 'Score', 'Metric': ''},  # Remove redundant axis titles
    notched=False,  # Avoid notched boxes (can be misleading)
    hover_data=['Score'],
    color_discrete_sequence=px.colors.qualitative.Safe,  # Professional color palette
)

# --- Layout Customization (Journal-Quality) ---
fig.update_layout(
    # Figure size (optimized for A4 subfigure)
    width=600,  # Slightly smaller to fit as subfigure
    height=450,
    
    # Font settings (must be legible at small sizes)
    font=dict(
        family="Arial, sans-serif",  # Standard academic font
        size=12,  # Base font size
        color="black"
    ),
    
    # Axis formatting
    xaxis=dict(
        title="Score",  # Remove x-axis title (implied by tick labels)
        title_font=dict(size=14),
        tickfont=dict(size=13),
        linecolor='black',
        showgrid=False,  # No gridlines on x-axis
    ),
    yaxis=dict(
        title=None,
        title_font=dict(size=14),
        tickfont=dict(size=13),
        range=[0, 1.05],  # Slight buffer above 1.0
        linecolor='black',
        gridcolor='lightgrey',  # Subtle gridlines
        gridwidth=0.5,
        minor_griddash="dot",
    ),
    
    # Boxplot aesthetics
    boxmode='group',
    showlegend=False,  # No legend (metrics labeled on x-axis)
    boxgap=0.2,  # Slightly wider boxes
    boxgroupgap=0.2,
    
    # White background (journal standard)
    plot_bgcolor='white',
    paper_bgcolor='white',
    
    # Margins (ensure no clipping)
    margin=dict(l=10, r=10, t=40, b=10),
)

# --- Box Styling ---
fig.update_traces(
    width=0.6,  # Wider boxes
    line=dict(width=1.5),  # Bold box outlines
    marker=dict(
        size=4,  # Smaller points
        opacity=0.5,  # Slightly transparent
        line=dict(width=0.5, color='black')  # Point borders
    ),
    jitter=0.2,  # Spread points slightly
)

# # --- Add mean markers (optional) ---
# for i, metric in enumerate(df_melted['Metric'].unique()):
#     mean_val = df_melted[df_melted['Metric'] == metric]['Score'].mean()
#     fig.add_shape(
#         type='line',
#         x0=i-0.25, x1=i+0.25,
#         y0=mean_val, y1=mean_val,
#         line=dict(color='black', width=1.5, dash='dash'),
#     )

# --- Save as vector graphic (for submission) ---
fig.write_image(os.path.join(save_dir,"boxplot_metrics.pdf"), format='pdf', scale=2)  # High-res PDF
fig.write_image(os.path.join(save_dir,"boxplot_metrics.svg"), format='svg', scale=2)
fig.write_html(os.path.join(save_dir,"boxplot_metrics.html"))
fig.show()

In [ ]:
# Check for best model:
best_model=df_classification_results["Cohen\'s Kappa"].reset_index(drop=True).idxmax()
best_model

In [ ]:
# Save best model
xgb_model_path = os.path.join(wd,f'models/{date}_kirc_classification_all_genes')
os.makedirs(xgb_model_path, exist_ok=True)
model_real_all_genes[best_model].save_model(os.path.join(xgb_model_path,'xgboost_model.json'))

## Classification along a trajectory

In [ ]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '112'
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xgboost as xgb

In [ ]:
wd = '/mnt/c/Repos/evenflow_cancer'

In [ ]:
import xgboost as xgb
date = '20250919'
xgb_model_path = os.path.join(wd,f'models/{date}_kirc_classification_all_genes')
saved_model = xgb.Booster()
saved_model.load_model(os.path.join(xgb_model_path,'xgboost_model.json'))

In [ ]:
path_save_figs = os.path.join(wd,f'reports/figures/{date}_KIRC_classification_along_trajectory')
os.makedirs(path_save_figs,exist_ok=True)

In [ ]:
# Function to load trajectory gene expression data
# Faster than exporting the whole dataframe at once
def load_traj_gene_expr(path,transition='early_to_late'):
    df_gene = pd.DataFrame()
    t_id = 0
    for csv_i in tqdm(next(os.walk(path))[2][:100],desc=f'Patient trajectory for transition {transition}'):
        path_csv_i = os.path.join(path,csv_i)
        df_gene_i = pd.read_csv(path_csv_i,index_col=0).T
        df_gene_i.insert(0,'ID',[t_id]*df_gene_i.shape[0])
        df_gene_i.insert(1,'Trajectory',[csv_i[:-4]]*df_gene_i.shape[0])
        df_gene_i.insert(2,'Transition',[transition]*df_gene_i.shape[0])
        df_gene = pd.concat([df_gene,df_gene_i],axis=0)
        t_id += 1
    return df_gene

In [ ]:
path_synth=os.path.join(wd,'data/interim/trajectories/early_to_late')
traj_gene = load_traj_gene_expr(path_synth,transition='early_to_late')
print(traj_gene.shape)

In [ ]:
df_data = traj_gene.copy()
model = saved_model
df_predictions_all = pd.DataFrame()
all_traj_ids = traj_gene.ID.unique()
for row_idx, traj_id in tqdm(enumerate(all_traj_ids, start=1),desc="Making predictions..."):
    df_plot = df_data[df_data['ID'] == traj_id]
    traj_i = df_plot['Trajectory'].unique()[0]
    new_data = df_plot.drop(['ID', 'Trajectory', 'Transition'], axis=1).T
    dnew = xgb.DMatrix(new_data)
    predictions_i = model.predict(dnew)
    df_predictions_i = pd.DataFrame(
        predictions_i, 
        index=new_data.index, 
        columns=['early', 'late']
    )
    df_predictions_i = df_predictions_i.melt(value_vars=['early', 'late'],var_name='Stage', value_name='Probability')
    df_predictions_i.insert(0,"Traj_ID",traj_id)
    df_predictions_i.insert(1,"Interpol_Index",[i for i in range(50)]*2) # list like 0,1,2,3 ... 48,49,0,1,2,...,
    df_predictions_all = pd.concat([df_predictions_all,df_predictions_i],axis=0)

print(df_predictions_all.shape)

In [ ]:
# Create the boxplot
fig = px.box(
    df_predictions_all,
    x='Interpol_Index',
    y='Probability',
    color='Stage',
    points=False,  # Hide individual points for clarity
    color_discrete_sequence=['#377eb8', '#e41a1c']  # Colorblind-friendly
)
# # Create the boxplot
# fig = px.box(df_predictions_all, x='Interpol_Index', y='Probability', color='Stage', 
#              points=False,  # show all points
#              title='Distribution of Early and Late by Interpol_Index')

fig.update_layout(
    width=1600,
    height=600,
    font=dict(family='Times New Roman', size=24),
    legend=dict(
        title='Stage',
        font=dict(size=22),
        bordercolor='black',
        borderwidth=1
    ),
    xaxis=dict(
        title=dict(text='Interpolation Index', font=dict(size=26)),
        tickfont=dict(size=20),
        showgrid=False,
        linecolor='black',
        mirror=True,
        tickvals=list(range(0, 50)),  # Show all ticks from 0 to 49
        range=[-0.5, 49.5]            # Ensure the last tick (49) is visible
    ),
    yaxis=dict(
        title=dict(text='Probability', font=dict(size=26)),
        tickfont=dict(size=20),
        showgrid=False,
        linecolor='black',
        mirror=True
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=80, r=40, t=20, b=80),
    title=None
)


# Remove the figure title (add in caption, not on figure)
fig.update_layout(title=None)
fig.write_html(os.path.join(path_save_figs, 'combined_trajectories_classification.html'))
fig.write_image(os.path.join(path_save_figs, 'combined_trajectories_classification.png'))
fig.write_image(os.path.join(path_save_figs, 'combined_trajectories_classification.pdf'),scale=2)
fig.write_image(os.path.join(path_save_figs, 'combined_trajectories_classification.svg'))
# Show the figure
fig.show()

### Classification along control trajectories (early_to_early and late_to_late)

In [ ]:
noise_in_real = pd.read_csv(
    os.path.join(wd,f'data/interim/trajectories/early_to_early/examples.csv'),
    index_col=0)
noise_in_real.shape

In [ ]:
# Format dataframe for classification
# Declare trajectory names (patient_to_patient)
trajectory = [i+'_to_'+j for (i,j) in zip(noise_in_real.source,noise_in_real.target)]
# Declare type of transitions (early_to_early)
transitions=[i+'_to_'+i for i in stages[noise_in_real.source]]
# Create trajectory IDs
N = 50 # number of timepoints
size = noise_in_real.shape[0]//N # number of trajectories
xss=np.array([[i]*N for i in range(size)]) # 50 timepoints, 530 trajectories
traj_ids=xss.reshape(noise_in_real.shape[0]) # flatten
# Change columns
noise_in_real.drop(['source','target','index'],inplace=True,axis=1)
noise_in_real.insert(0,'ID',traj_ids)
noise_in_real.insert(1,'Trajectory',trajectory)
noise_in_real.insert(2,'Transition',transitions)
# noise_in_real.rename(columns={'index':'Interpol_Index'}, inplace=True)

In [ ]:
date = "20250915"
traj_type = "early_to_early"
# traj_type = "late_to_late"

path_save_figs = os.path.join(wd,f'reports/figures/{date}_KIRC_classification_along_trajectory',traj_type)
os.makedirs(path_save_figs, exist_ok=True)

traj_gene = noise_in_real.copy()
traj_gene = traj_gene[traj_gene['Transition']==traj_type]

In [ ]:
df_data = traj_gene.copy()
df_data = noise_in_real.copy()
model = saved_model
df_predictions_all = pd.DataFrame()
all_traj_ids = traj_gene.ID.unique()
# all_traj_ids = noise_in_real.ID.unique()
for row_idx, traj_id in tqdm(enumerate(all_traj_ids, start=1),desc="Making predictions..."):
    df_plot = df_data[df_data['ID'] == traj_id]
    traj_i = df_plot['Trajectory'].unique()[0]
    new_data = df_plot.drop(['ID', 'Trajectory', 'Transition'], axis=1)
    dnew = xgb.DMatrix(new_data)
    predictions_i = model.predict(dnew)
    df_predictions_i = pd.DataFrame(
        predictions_i,
        index=new_data.index,
        columns=['early', 'late']
    )
    df_predictions_i = df_predictions_i.melt(value_vars=['early', 'late'],var_name='Stage', value_name='Probability')
    df_predictions_i.insert(0,"Traj_ID",traj_id)
    df_predictions_i.insert(1,"Interpol_Index",[i for i in range(50)]*2) # list like 0,1,2,3 ... 48,49,0,1,2,...,
    df_predictions_all = pd.concat([df_predictions_all,df_predictions_i],axis=0)

print(df_predictions_all.shape)

In [ ]:
# Create the boxplot
fig = px.box(
    df_predictions_all,
    x='Interpol_Index',
    y='Probability',
    color='Stage',
    points=False,  # Hide individual points for clarity
    color_discrete_sequence=['#377eb8', '#e41a1c']  # Colorblind-friendly
)
# # Create the boxplot
# fig = px.box(df_predictions_all, x='Interpol_Index', y='Probability', color='Stage',
#              points=False,  # show all points
#              title='Distribution of Early and Late by Interpol_Index')

fig.update_layout(
    width=1600,
    height=600,
    font=dict(family='Times New Roman', size=24, color='black'),
    legend=dict(
        title=dict(text='Stage', font=dict(size=22, color='black')),
        font=dict(size=22, color='black'),
        bordercolor='black',
        borderwidth=1
    ),
    xaxis=dict(
        title=dict(text='Interpolation Index', font=dict(size=26, color='black')),
        tickfont=dict(size=20, color='black'),
        showgrid=False,
        linecolor='black',
        mirror=True,
        tickvals=list(range(0, 50)),  # Show all ticks from 0 to 49
        range=[-0.5, 49.5]            # Ensure the last tick (49) is visible
    ),
    yaxis=dict(
        title=dict(text='Probability', font=dict(size=26, color='black')),
        tickfont=dict(size=20, color='black'),
        showgrid=False,
        linecolor='black',
        mirror=True
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=80, r=40, t=20, b=80),
    title=None
)


# Remove the figure title (add in caption, not on figure)
fig.update_layout(title=None)
fig.write_html(os.path.join(path_save_figs, 'combined_trajectories_classification.html'))
fig.write_image(os.path.join(path_save_figs, 'combined_trajectories_classification.png'))
fig.write_image(os.path.join(path_save_figs, 'combined_trajectories_classification.pdf'),scale=2)
fig.write_image(os.path.join(path_save_figs, 'combined_trajectories_classification.svg'))
# Show the figure
fig.show()